# Machine Learning in Python - Project 

Due Friday, December 4th 2026, by 4 pm.

**Finn McLachlan, Thomas Sinclair, Sridhar, Bagus Awan**

## Setup

*Install any packages here, define any functions if needed, and load data.*

In [1]:
# Add any additional libraries or submodules below

# Data libraries
import pandas as pd
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn modules
import sklearn

In [2]:
# Load data  
df_avg = pd.read_csv("avg_river_levels.csv")
df_min = pd.read_csv("min_river_levels.csv")
df_max = pd.read_csv("max_river_levels.csv")
df_locs = pd.read_csv("monitoring_locations.csv")
df_avg.head()


,County,Station_Name,Observation_ID,05/10/2014,06/10/2014,07/10/2014,08/10/2014,09/10/2014,10/10/2014,13/10/2014,...,29/09/2026,30/09/2026,01/10/2026,02/10/2026,03/10/2026,04/10/2026,05/10/2026,06/10/2026,07/10/2026,08/10/2026
0,aberdeenshire,"Bogie, Redcraig",aberdeenshire__bogie-redcraig__obs1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.133,0.152,0.169,0.143,0.136,0.130,0.126,0.120,0.120,0.120
1,aberdeenshire,"Dee (Grampian), Mar Lodge",aberdeenshire__dee-grampian-mar-lodge__obs1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.721,0.971,0.849,0.799,0.861,0.752,0.727,0.695,0.677,0.669
2,aberdeenshire,"Dee (Grampian), Park",aberdeenshire__dee-grampian-park__obs1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.934,0.973,1.264,0.990,1.054,0.988,0.918,0.887,0.854,0.836
3,aberdeenshire,"Dee (Grampian), Polhollick",aberdeenshire__dee-grampian-polhollick__obs1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.483,0.704,0.683,0.530,0.676,0.530,0.496,0.460,0.438,0.427
4,aberdeenshire,"Dee (Grampian), Woodend",aberdeenshire__dee-grampian-woodend__obs1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.578,0.700,0.861,0.645,0.739,0.642,0.583,0.547,0.514,0.495


# Introduction

*This section should include a brief introduction to the task and the data (assume this is a report you are delivering to a professional body (e.g. Scottish Government)).*

*State which supervised learning question you are answering.*

*Briefly outline the approaches being used (for both the clustering and your chosen supervised learning task) and the conclusions that you are able to draw.*

We have been tasked with performing an investigation into the river water levels, and whether predictions can be made about water levels in future. 

# Exploratory Data Analysis and Feature Engineering

*Include a detailed discussion of the data with a particular emphasis on the features of the data that are relevant for the subsequent modeling. Including visualisations of the data is strongly encouraged - all code and plots must also be described in the write up. Think carefully about whether each plot needs to be included in your final draft and the appropriate type of plot and summary for each variable type - your report should include figures but they should be as focused and impactful as possible.*

*You should also split your data into training and testing sets, ideally before you look too much into the features and relationships with the target, as discussed throughout the course.*

*Additionally, this section should also motivate and describe any preprocessing / feature engineering of the data. Pipelines should be used, and feature engineering steps that are performed as part of an sklearn pipeline should be implemented in the following section.*

*All code and figures should be accompanied by text that provides an overview / context to what is being done or presented.*

In [7]:
df_min.shape[0],df_avg.shape[0],df_max.shape[0] #notice they are all the same length

df_min.info()
#df

<class 'pandas.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Columns: 4382 entries, County to 08/10/2026
dtypes: float64(4379), str(3)
memory usage: 13.1 MB


In [4]:
#elongate df to put in better form

df_avg_long = df_avg.melt(
    id_vars=["County","Station_Name","Observation_ID"],         
    var_name="Date",           
    value_name="Avg_Level"    
)

df_min_long = df_min.melt(
    id_vars=["County","Station_Name","Observation_ID"],        
    var_name="Date",           
    value_name="Min_Level"         
)
df_max_long = df_max.melt(
    id_vars=["County","Station_Name","Observation_ID"],         
    var_name="Date",           
    value_name="Max_Level"         
)

#combine if certain values are equal (just in case)
df_main = (
    df_min_long
    .merge(df_max_long, on=["Observation_ID","County","Station_Name","Date"], how="inner")
    .merge(df_avg_long, on=["Observation_ID","County","Station_Name","Date"], how="inner")
)

#drop nas if min max and avg are na
df_main = df_main.dropna(subset=["Min_Level", "Max_Level", "Avg_Level"], how="all")

In [5]:
df_main.info()
df_main.head()

<class 'pandas.DataFrame'>
Index: 1190691 entries, 122 to 1716567
Data columns (total 7 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   County          1190691 non-null  str    
 1   Station_Name    1190691 non-null  str    
 2   Observation_ID  1190691 non-null  str    
 3   Date            1190691 non-null  str    
 4   Min_Level       1190691 non-null  float64
 5   Max_Level       1190691 non-null  float64
 6   Avg_Level       1190691 non-null  float64
dtypes: float64(3), str(4)
memory usage: 72.7 MB


,County,Station_Name,Observation_ID,Date,Min_Level,Max_Level,Avg_Level
122,dumfries,"River Esk, Canonbie (SEPA Site)",dumfries__esk-border-dumfries-and-galloway-can...,05/10/2014,0.92,1.09,1.005
301,"roxburgh,-ettrick-and-lauderdale","River Tweed, Sprouston","roxburgh,-ettrick-and-lauderdale__tweed-scotti...",05/10/2014,1.40,1.40,1.400
514,dumfries,"River Esk, Canonbie (SEPA Site)",dumfries__esk-border-dumfries-and-galloway-can...,06/10/2014,0.82,2.05,1.620
693,"roxburgh,-ettrick-and-lauderdale","River Tweed, Sprouston","roxburgh,-ettrick-and-lauderdale__tweed-scotti...",06/10/2014,0.97,0.97,0.970
906,dumfries,"River Esk, Canonbie (SEPA Site)",dumfries__esk-border-dumfries-and-galloway-can...,07/10/2014,0.97,1.13,1.050


In [ ]:
# Split into train and test sets For clustering we will need to predict Observation id
# Are we going to try cluster by station or county? I think we should cluster by county and then see if we can predict the station name. Could be interesting to look at both
# Taken from WS8

# Feature matrix and response vector
#X, y = penguins_rm.drop(["island", "sex", "species"], axis=1), penguins_rm['Observation_ID']

# Stratify split
#X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle= True, test_size = 0.2, random_state=0)

# Model Fitting and Tuning

*In this section you should detail and motivate your choice of model and describe the process used to refine, tune, and fit that model. You are encouraged to explore different models but you should NOT include a detailed narrative or code of all of these attempts. At most this section should very briefly mention the methods explored and why they were rejected - most of your effort should go into describing the final model you are using and your process for tuning and validating it.*

*This section should include the full implementation of your final model, including all necessary validation. All displayed figures should be described and referenced in the text.*

*You should also include a baseline model of your choice and provide a comparison of your model with the baseline model on the test data. You should briefly describe the baseline model considered.*

# Interpretation, Discussion & Conclusions

*In this section you should provide a general overview of your final clustering and of your final supervised learning model, its performance, and reliability. You should discuss what the implications of your models are in terms of the included features, estimated parameters and relationships, predictive performance, and anything else you think is relevant.*

*This should be written with a target audience of a government official, who understands the monitoring of river levels but may only have university level mathematics (not necessarily postgraduate statistics or machine learning). Your goal should be to highlight to this audience how your model can useful. You should also discuss potential limitations or directions of future improvement of your model.*

*Finally, you should include recommendations on factors that may influence the classification of flood events OR the prediction of future water levels (depending on which question you are answering), which may be useful for the government officials to improve their understanding of floods or water level forecasting, and potentially assist in the development of effective river management and interventions.*

*Keep in mind that a negative result, i.e. a model that does not work well predictively, that is well explained and justified in terms of why it failed will likely receive higher marks than a model with strong predictive performance but with poor or incorrect explanations / justifications.*

# Generative AI statement

*Include a statement on if and how generative AI was used in the project and report.*

# References

*Include a list of all references cited in the text.*

In [1]:
# Run the following to render to PDF
!jupyter nbconvert --to pdf project.ipynb

[NbConvertApp] Converting notebook project.ipynb to pdf
[NbConvertApp] Writing 29863 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | bibtex had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 47131 bytes to project.pdf
